# 15 - BatchNorm ablation: does BatchNormalization improve MiniConvNet?

**A controlled architecture ablation, not a re-optimisation.** Given this project's documented
history of training instability (dead-ReLU collapse, partial collapse), BatchNorm is a well-motivated,
standard technique to test - not a random guess. This notebook adds **exactly one** architectural
change to MiniConvNet - `BatchNormalization` after each `Conv2D`, before its activation - and measures
whether it helps, hurts, or makes no distinguishable difference, using the identical evaluation setup
as **Option B** (`notebooks/14_leakage_controlled_cv.ipynb`) so the comparison is apples-to-apples.

**Whatever this comes back as gets reported as-is.** BatchNorm doing nothing, or making things
slightly worse, is exactly as legitimate an ablation result as an improvement - see Step 5.

## What already exists - checked before adding anything

`src/models.py` -> `build_miniconvnet()` **already has a `use_batchnorm` parameter**, defaulting to
`False`, and it is **already placed exactly as this ablation requires**:

```python
x = layers.Conv2D(f, 3, padding="same", activation=None, ...)(x)
if use_batchnorm:
    x = layers.BatchNormalization(name=f"bn{i}")(x)
x = _activation(f"act{i}", activation)(x)          # LeakyReLU
x = layers.MaxPooling2D(2, name=f"pool{i}")(x)
```

applied in all 4 conv blocks, and **nowhere else** (not after `Dense(64)`, not in the head at all).
**No change to `src/models.py` was needed or made** - this notebook only ever passes
`use_batchnorm=True` at the call site; the default (`False`) used by every other notebook in this
project is completely untouched.

## What is reused, not duplicated

`src/leakage_cv_utils.py` (Option B's own module) is reused directly:
`load_leakage_controlled_split()`, `assert_no_group_leakage_in_folds()`, and
`write_leakage_controlled_cv_results()` are called unmodified in their behaviour. Two small additions
were made to that module (both additive, both backward-compatible - see the module's own diff):

* `load_option_b_reference()` / `compare_to_option_b()` - Option B's own leakage-controlled result
  (74.10% +/- 4.24%, read live from `outputs/leakage/leakage_controlled_cv_results.json`, not
  hardcoded) is the correct comparison point for this ablation, not the original pooled-CV headline.
* `write_leakage_controlled_cv_results()` gained optional `csv_path`/`json_path`/`task_label`/
  `purpose_text` keyword arguments, all defaulting to `None` -> Option B's own files, exactly as
  before. This notebook is the only caller that supplies non-default values (pointing at
  `leakage_controlled_cv_batchnorm_results.csv`/`.json` instead), so Option B's own output files are
  never at risk of being overwritten by this notebook.

**The evaluation pipeline itself - `StratifiedGroupKFold` on `image_group`, the identical training
configuration, `detect_collapse()`/`detect_partial_collapse()` on every fold - is copied verbatim from
`notebooks/14_leakage_controlled_cv.ipynb`.** The single line that differs in the training loop is the
model-building call: `build_miniconvnet(use_batchnorm=True)` instead of `build_miniconvnet()`.

**Nothing in this notebook has been executed.** It was written, not run.

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

from src.config import *
from src.data_utils import resolve_data_root

ensure_dirs()
print('data root:', resolve_data_root())
print('folds:', CV_FOLDS, '| epochs per fold:', EPOCHS_CV, '(identical to Option B / notebooks/03)')

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import StratifiedGroupKFold, train_test_split

from src.data_utils import make_dataset
from src.models import build_miniconvnet, count_params
from src.train_utils import (set_global_seeds, compute_report, compile_model, optimizer_summary,
                             make_callbacks, make_epoch_timer, save_history, final_epoch_summary,
                             run_name_for, estimate_training_time)
from src.evaluate_utils import (predict, compute_metrics, detect_collapse, print_collapse_report,
                                confusion, summarize_cv, format_mean_std, tumor_vs_subtype_breakdown,
                                interpret_breakdown, save_predictions, plot_confusion_matrix)

from src.leakage_cv_utils import (
    LEAKAGE_CONTROLLED_CV_BATCHNORM_RESULTS_CSV, LEAKAGE_CONTROLLED_CV_BATCHNORM_RESULTS_JSON,
    load_leakage_controlled_split, assert_no_group_leakage_in_folds,
    load_option_b_reference, compare_to_option_b, write_leakage_controlled_cv_results,
)

set_global_seeds(SEED)
for k, v in compute_report().items():
    print(f'{k}: {v}')

## STEP 1 - architecture: build the BatchNorm variant and report its real parameter count

**Nothing here trains anything** - `build_miniconvnet(use_batchnorm=True)` only constructs the Keras
graph. `model.summary()` and `count_params()` report the actual, measured parameter count, not an
estimate - print this before running anything so the "lightweight" framing is verified rather than
assumed to still hold.

For context (computed by hand, NOT a substitute for the printed measurement below): BatchNorm adds
`gamma` and `beta` (trainable) plus `moving_mean`/`moving_variance` (non-trainable) per channel, per
layer. With filters `(16, 32, 64, 128)` across the 4 conv blocks, that is `2 x (16+32+64+128) = 480`
new trainable parameters and `480` new non-trainable parameters - an expected total near
`499,172 + 960 = 500,132`, still comfortably in the same "lightweight" order of magnitude as the
original. The cell below reports what the model actually has, not this estimate.

In [ ]:
bn_model = build_miniconvnet(use_batchnorm=True)
bn_model.summary()

In [ ]:
bn_params = count_params(bn_model)
base_params = count_params(build_miniconvnet(use_batchnorm=False))   # for the honest side-by-side

print('BatchNorm variant parameter count:')
for k, v in bn_params.items():
    print(f'  {k}: {v}')
print()
print('vs. the existing MiniConvNet (use_batchnorm=False, the project standard):')
for k, v in base_params.items():
    print(f'  {k}: {v}')
print()
added_total = bn_params['total_params'] - base_params['total_params']
added_trainable = bn_params['trainable_params'] - base_params['trainable_params']
print(f"BatchNorm added {added_total:,} total parameters ({added_trainable:,} trainable, "
      f"{added_total - added_trainable:,} non-trainable) - confirm this matches the hand-computed "
      'estimate above (960 total, 480/480 split) before trusting anything downstream.')

del bn_model
tf.keras.backend.clear_session()

## STEP 1 (cont.) - load the leakage-controlled split (identical to Option B)

Same load-and-verify step as Option B - does not construct a new split. If this fails, the notebook
stops here exactly as Option B's does.

In [ ]:
load_result = load_leakage_controlled_split()

SPLIT_LOADED_OK = load_result['ok']
if not SPLIT_LOADED_OK:
    print('=' * 78)
    print('STOP - the leakage-controlled split could not be loaded.')
    print('=' * 78)
    print(load_result['reason'])
else:
    pool = load_result['df']
    print(f"loaded: {load_result['path']}")
    print(f'pooled images: {len(pool)}')
    print(pool['class'].value_counts().reindex(CLASS_NAMES).to_string())

## STEP 2 - fold construction: identical `StratifiedGroupKFold` as Option B

Same seed, same `n_splits`, same `groups=image_group` - the fold membership used here is the same
kind of construction Option B used (not necessarily numerically identical fold-for-fold, since this
call re-derives it, but built with the same code and the same guarantee, verified below exactly as
Option B verified it).

In [ ]:
if SPLIT_LOADED_OK:
    sgkf = StratifiedGroupKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)
    folds = list(sgkf.split(pool.index.values, pool['label'].values,
                            groups=pool['image_group'].values))

    for i, (tr_idx, te_idx) in enumerate(folds, start=1):
        te_counts = pool.iloc[te_idx]['class'].value_counts().reindex(CLASS_NAMES).to_dict()
        print(f'fold {i}: train={len(tr_idx):4d} test={len(te_idx):4d} test class counts={te_counts}')

    group_check = assert_no_group_leakage_in_folds(pool, folds)
    print('\nStratifiedGroupKFold group-leakage check:', group_check)
    assert group_check['clean'], (
        'image_group violations found - the leakage control failed for this run; investigate '
        'before proceeding, do not report results from a leaky CV.')
    print('CONFIRMED: no image_group straddles any fold\'s train/test boundary.')

## Run helper - IDENTICAL to Option B except `use_batchnorm=True`

Compare this cell against `notebooks/14_leakage_controlled_cv.ipynb`'s `run_fold()`: every line is
the same except the model-building call and the run-name/meta tags (so these files never collide
with Option B's own predictions/history).

In [ ]:
def run_fold(fold_idx, tr_idx, te_idx, verbose=2):
    set_global_seeds(SEED + fold_idx)   # identical to Option B - different init per fold, reproducible
    run_name = run_name_for('cv', 'leakage_controlled_batchnorm', f'fold{fold_idx}')

    train_pool = pool.iloc[tr_idx].reset_index(drop=True)
    test_df = pool.iloc[te_idx].reset_index(drop=True)
    # Inner train/val split for early stopping: identical to Option B (plain stratified split,
    # no group constraint).
    tr_df, val_df = train_test_split(train_pool, test_size=0.15,
                                     stratify=train_pool['class'],
                                     random_state=SEED + fold_idx)

    # one_hot=True because the model trains with label smoothing - identical to Option B.
    train_ds = make_dataset(tr_df, shuffle=True, augment=True, seed=SEED + fold_idx, one_hot=True)
    val_ds = make_dataset(val_df, one_hot=True)
    test_ds = make_dataset(test_df, one_hot=True)

    # THE ONE CHANGE UNDER TEST: use_batchnorm=True. Everything else - filters, extra_pool,
    # dense_units, dropout_rate, activation, kernel_initializer - is left at its project-standard
    # default, identical to Option B's build_miniconvnet() call.
    model = compile_model(build_miniconvnet(use_batchnorm=True), verbose=False)
    if fold_idx == 1:
        print('optimizer:', optimizer_summary(model))   # printed once, to confirm the training
                                                          # config itself is untouched

    timer = make_epoch_timer(verbose=0)
    history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_CV,
                        callbacks=make_callbacks(run_name, timer=timer, checkpoint=False,
                                                 verbose=0),
                        verbose=verbose)
    save_history(history, run_name, timer=timer)
    summary = final_epoch_summary(history, timer=timer)

    y_true, y_pred, y_prob = predict(model, test_ds)
    metrics = compute_metrics(y_true, y_pred, y_prob)

    save_predictions(run_name, y_true, y_pred, y_prob,
                     meta={'split_variant': 'leakage_controlled_image_group', 'fold': fold_idx,
                           'activation': ACTIVATION, 'label_smoothing': LABEL_SMOOTHING,
                           'use_batchnorm': True,
                           'purpose': 'BatchNorm ablation - compared against Option B, not the '
                                      'original headline'})

    collapse = detect_collapse(history=history, kappa=metrics['cohen_kappa'],
                               mcc=metrics['mcc'], y_pred=y_pred)
    breakdown = tumor_vs_subtype_breakdown(y_true, y_pred)
    n_classes = collapse['details'].get('n_predicted_classes')

    print(f"\nfold {fold_idx} -> acc={metrics['accuracy']:.4f} f1={metrics['f1_macro']:.4f} "
          f"kappa={metrics['cohen_kappa']:.4f} classes={n_classes}/{NUM_CLASSES} "
          f"epochs={summary['epochs_trained']} "
          f"time={summary.get('total_minutes')}min status={collapse['status']}")
    if collapse['collapsed']:
        print_collapse_report(collapse, run_name)
        print('confusion matrix:')
        print(confusion(y_true, y_pred))

    tf.keras.backend.clear_session()
    return {'fold': fold_idx, 'run_name': run_name, 'status': collapse['status'],
            'n_predicted_classes': n_classes,
            'epochs_trained': summary['epochs_trained'],
            'minutes': summary.get('total_minutes'),
            'binary_tumor_acc': breakdown['binary_tumor_vs_healthy_accuracy'],
            'subtype_acc': breakdown['subtype_accuracy_all_tumors'],
            'y_true': y_true, 'y_pred': y_pred, **metrics}

## Time estimate before committing to all 3 folds

BatchNorm adds a small forward/backward-pass cost per batch (extra normalisation ops) - measured,
not assumed, exactly like every training notebook in this project.

In [ ]:
if SPLIT_LOADED_OK:
    _probe_tr = make_dataset(pool.iloc[folds[0][0]].reset_index(drop=True),
                             shuffle=True, augment=True, seed=SEED, one_hot=True)
    _probe_va = make_dataset(pool.iloc[folds[0][1]].reset_index(drop=True), one_hot=True)

    est = estimate_training_time(
        model_fn=lambda: compile_model(build_miniconvnet(use_batchnorm=True), verbose=False),
        train_ds=_probe_tr, val_ds=_probe_va,
        planned_epochs=EPOCHS_CV, n_runs=CV_FOLDS)

    del _probe_tr, _probe_va

## STEP 2 (cont.) - run the 3 folds

In [ ]:
if SPLIT_LOADED_OK:
    cv_results = [run_fold(i, tr, te) for i, (tr, te) in enumerate(folds, start=1)]
    print('\nfolds complete:', len(cv_results))
    print('total wall clock:', round(sum(f['minutes'] or 0 for f in cv_results), 1), 'min')

## STEP 3 - per-fold metrics, and collapse exclusion (identical convention to every CV notebook)

Applying `detect_collapse()`/`detect_partial_collapse()` to every fold exactly as done for Option B
and the original headline - any collapsed fold is excluded from the mean, not silently averaged in.
BatchNorm's own running statistics add a plausible NEW way a fold could misbehave (unstable batch
statistics with a small per-fold batch count), so this check matters at least as much here as
anywhere else in the project.

In [ ]:
if SPLIT_LOADED_OK:
    cols = ['fold', 'accuracy', 'f1_macro', 'precision_macro', 'recall_macro', 'cohen_kappa', 'mcc',
            'binary_tumor_acc', 'subtype_acc', 'n_predicted_classes', 'epochs_trained', 'minutes',
            'status']
    fold_tbl = pd.DataFrame(cv_results)[cols]
    print(fold_tbl.round(4).to_string(index=False))

    bad = fold_tbl[fold_tbl['status'] != VALID_TAG]
    if len(bad):
        print(f"\n!!! {len(bad)} fold(s) invalid and excluded from the mean: "
              + ', '.join(f"fold {int(r.fold)} ({r.status})" for r in bad.itertuples()))
    else:
        print('\nAll folds valid.')

In [ ]:
if SPLIT_LOADED_OK:
    valid = [f for f in cv_results if f['status'] == VALID_TAG]
    dropped = len(cv_results) - len(valid)
    n_full = sum(1 for f in cv_results if f['status'] == COLLAPSE_TAG)
    n_partial = sum(1 for f in cv_results if f['status'] == PARTIAL_COLLAPSE_TAG)

    if not valid:
        print('!!! All folds are invalid - there is nothing to report. This is itself a finding: '
              'write it up with the per-fold confusion matrices above (BatchNorm may be actively '
              'harmful at this batch size/dataset size), do not retry blindly.')
        batchnorm_summary = None
    else:
        batchnorm_summary_raw = summarize_cv(valid)
        bn_params_final = count_params(build_miniconvnet(use_batchnorm=True))['total_params']
        tf.keras.backend.clear_session()

        print(f"{len(valid)}/{len(cv_results)} valid folds "
              f"({dropped} excluded: {n_full} collapsed, {n_partial} partially collapsed)")
        for k in ('accuracy', 'f1_macro', 'cohen_kappa', 'mcc'):
            print(f"  {k}: {format_mean_std(batchnorm_summary_raw[k + '_mean'], batchnorm_summary_raw[k + '_std'])}")
        if dropped:
            print(f'  NOTE: this is the mean over {len(valid)} folds, not {len(cv_results)}. '
                  'Quote n_runs whenever you quote the number.')

        batchnorm_summary = {
            'accuracy_mean': batchnorm_summary_raw['accuracy_mean'],
            'accuracy_std': batchnorm_summary_raw['accuracy_std'],
            'f1_macro_mean': batchnorm_summary_raw['f1_macro_mean'],
            'f1_macro_std': batchnorm_summary_raw['f1_macro_std'],
            'cohen_kappa_mean': batchnorm_summary_raw['cohen_kappa_mean'],
            'mcc_mean': batchnorm_summary_raw['mcc_mean'],
            'binary_tumor_acc_mean': float(np.mean([f['binary_tumor_acc'] for f in valid])),
            'subtype_acc_mean': float(np.mean([f['subtype_acc'] for f in valid])),
            'n_folds_valid': len(valid), 'n_folds_total': len(cv_results),
            'n_folds_excluded_collapsed': n_full, 'n_folds_excluded_partial_collapse': n_partial,
            'params': bn_params_final,
        }

## Pooled confusion matrix across valid folds

In [ ]:
if SPLIT_LOADED_OK and batchnorm_summary is not None:
    y_true_all = np.concatenate([f['y_true'] for f in valid])
    y_pred_all = np.concatenate([f['y_pred'] for f in valid])
    print(f'pooled over {len(valid)} folds, {len(y_true_all)} predictions')

    save_predictions('cv_leakage_controlled_batchnorm_pooled', y_true_all, y_pred_all,
                     meta={'split_variant': 'leakage_controlled_image_group',
                           'source': f'pooled over {len(valid)} BatchNorm-ablation CV folds',
                           'use_batchnorm': True,
                           'purpose': 'BatchNorm ablation - compared against Option B'})
    plot_confusion_matrix(y_true_all, y_pred_all, 'cv_leakage_controlled_batchnorm_pooled')

    pooled_breakdown = tumor_vs_subtype_breakdown(y_true_all, y_pred_all)
    for k, v in pooled_breakdown.items():
        print(f'  {k}: {v}')
    print()
    print(interpret_breakdown(pooled_breakdown))

## STEP 4 - direct comparison against Option B (74.10% +/- 4.24%)

Read live from `outputs/leakage/leakage_controlled_cv_results.json` (never a hardcoded number,
unless that file happens to be absent on this machine, in which case a documented fallback matching
the value stated in the task brief is used and clearly labelled as such).

In [ ]:
if SPLIT_LOADED_OK and batchnorm_summary is not None:
    option_b = load_option_b_reference()
    print('OPTION B REFERENCE (read live):')
    for k, v in option_b.items():
        print(f'  {k}: {v}')

    comparison = compare_to_option_b(batchnorm_summary, option_b)
    print()
    print('SIDE-BY-SIDE COMPARISON')
    print('-' * 50)
    print(f"  BatchNorm accuracy : {comparison['batchnorm_accuracy_mean']:.4f} "
          f"+/- {comparison['batchnorm_accuracy_std']:.4f}")
    print(f"  Option B accuracy  : {comparison['option_b_accuracy_mean']:.4f} "
          f"+/- {comparison['option_b_accuracy_std']:.4f}")
    print(f"  difference         : {comparison['difference']:+.4f}")
    print(f"  within Option B's own std? : {comparison['within_option_b_std_dev']}")
    print()
    print(comparison['verdict'])
    print()
    print('Same comparison, other metrics:')
    print(f"  f1_macro   : BatchNorm={batchnorm_summary['f1_macro_mean']:.4f} "
          f"vs Option B={option_b['f1_macro_mean']:.4f}")
    print(f"  tumour det.: BatchNorm={batchnorm_summary['binary_tumor_acc_mean']:.4f} "
          f"vs Option B={option_b['binary_tumor_acc']:.4f}")
    print(f"  subtype    : BatchNorm={batchnorm_summary['subtype_acc_mean']:.4f} "
          f"vs Option B={option_b['subtype_acc']:.4f}")

## STEP 6 - save results (new files only - Option B's own results are never touched)

Writes to `outputs/leakage/leakage_controlled_cv_batchnorm_results.csv`/`.json` via
`write_leakage_controlled_cv_results()`'s new optional `csv_path`/`json_path` arguments - the same
function Option B uses, redirected to its own files rather than Option B's.

In [ ]:
if SPLIT_LOADED_OK and batchnorm_summary is not None:
    paths = write_leakage_controlled_cv_results(
        per_fold_df=fold_tbl, summary=batchnorm_summary, comparison=comparison,
        group_leakage_check=group_check,
        extra={'pooled_confusion_matrix': confusion(y_true_all, y_pred_all).tolist(),
               'pooled_breakdown': pooled_breakdown,
               'parameter_count_comparison': {
                   'batchnorm_total_params': bn_params['total_params'],
                   'baseline_total_params': base_params['total_params'],
                   'added_total_params': bn_params['total_params'] - base_params['total_params'],
                   'added_trainable_params': bn_params['trainable_params'] - base_params['trainable_params'],
               }},
        csv_path=LEAKAGE_CONTROLLED_CV_BATCHNORM_RESULTS_CSV,
        json_path=LEAKAGE_CONTROLLED_CV_BATCHNORM_RESULTS_JSON,
        task_label='BatchNorm ablation for MiniConvNet (3-fold CV, leakage-controlled split)',
        purpose_text=('Test whether adding BatchNormalization (after each Conv2D, before its '
                     'activation, in all 4 conv blocks - and nowhere else) improves training '
                     'stability or accuracy vs. the project-standard MiniConvNet. Compared '
                     'against Option B\'s own leakage-controlled result, not the original '
                     'pooled-CV headline. NOT an attempt to chase a better number regardless '
                     'of architecture correctness.'))
    print('files written:')
    for k, v in paths.items():
        print(f'  {k}: {v}')
    print()
    print("Option B's own files were NOT modified by this notebook - its accuracy on record "
          f"remains {option_b['accuracy_mean']:.4f} +/- {option_b['accuracy_std']:.4f} "
          f"(re-read live above, from {option_b['source']}).")

## STEP 5 - interpretation

Fill in once the cells above have actually been run. **Report the actual result regardless of
direction - BatchNorm doing nothing, or making things slightly worse, is a legitimate ablation
result, not a failed experiment.**

- Did BatchNorm's result fall inside or outside Option B's own standard deviation (Step 4)?
- If outside and better: BatchNorm is evidence-backed as a genuine improvement for this architecture -
  say so, and note the new parameter count (Step 1) so the "lightweight" framing stays accurate.
- If outside and worse: state plainly that BatchNorm measurably hurts this architecture at this
  dataset/batch size - a plausible explanation is noisy running-statistics estimates with small
  per-fold batch counts, but state what was actually observed (collapse counts, confusion matrix
  shape) rather than asserting a mechanism without evidence for it.
- If inside (within one std dev): BatchNorm is neither clearly helping nor clearly hurting - report
  this as inconclusive-but-informative, not as "no effect" (a single 3-fold estimate has limited
  power to detect a small effect either way).
- Cross-check against the collapse-detection results specifically: did BatchNorm change how many
  folds collapsed or partially collapsed, even if the mean accuracy among *valid* folds looks similar?
  That would be evidence about training *stability* distinct from evidence about final accuracy - the
  original motivating question for this ablation.

In [ ]:
if SPLIT_LOADED_OK and batchnorm_summary is not None:
    print('BATCHNORM PARAMETER COUNT:', end=' ')
    print(f"{bn_params['total_params']:,} total ({bn_params['trainable_params']:,} trainable, "
          f"+{bn_params['total_params'] - base_params['total_params']:,} vs. the {base_params['total_params']:,}-param baseline)")
    print('BATCHNORM ACCURACY (mean +/- std):        ', end=' ')
    print(f"{comparison['batchnorm_accuracy_mean']:.4f} +/- {comparison['batchnorm_accuracy_std']:.4f}")
    print('OPTION B ACCURACY (mean +/- std, no BN):  ', end=' ')
    print(f"{comparison['option_b_accuracy_mean']:.4f} +/- {comparison['option_b_accuracy_std']:.4f}")
    print(f"DIFFERENCE:                                {comparison['difference']:+.4f}")
    print(f"WITHIN OPTION B STD DEV?:                   {comparison['within_option_b_std_dev']}")
    print(f"FOLDS EXCLUDED FOR COLLAPSE:                 {batchnorm_summary['n_folds_excluded_collapsed']} "
          f"collapsed, {batchnorm_summary['n_folds_excluded_partial_collapse']} partially collapsed "
          f"(of {batchnorm_summary['n_folds_total']} total)")
    print('FILES CREATED:')
    print(f'  {LEAKAGE_CONTROLLED_CV_BATCHNORM_RESULTS_CSV}')
    print(f'  {LEAKAGE_CONTROLLED_CV_BATCHNORM_RESULTS_JSON}')
    print('  outputs/predictions/cv_leakage_controlled_batchnorm_fold{1,2,3}_predictions.csv')
    print('  outputs/predictions/cv_leakage_controlled_batchnorm_pooled_predictions.csv')
    print('CPU TIME:                                   sum of per-fold minutes printed above')
    print(f"CONCLUSION:                                  {comparison['verdict']}")
elif SPLIT_LOADED_OK and batchnorm_summary is None:
    print('BATCHNORM ACCURACY (mean +/- std): n/a - all folds were invalid, see above')
    print('CONCLUSION: all folds collapsed or partially collapsed with BatchNorm enabled - this is')
    print('itself a strong, reportable finding (BatchNorm actively harmful here), not a bug to fix')
    print('by silently retrying.')
else:
    print('Nothing to report - the leakage-controlled split could not be loaded (see STEP 1 above).')